[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/STAT3009_Side.ipynb)

# Recommendation with Side Information

**STAT3009 · Recommender Systems**

> **Today:** start from the NCF `MFNet`, then add one side-information block at a time. Continuous features, categorical embeddings, text, and image all become small neural modules around the same MF base.

**Route:** setup (3 min) → prepare data (10 min) → MFNet baseline (10 min) → continuous (8 min) → categorical (8 min) → text (10 min) → image (8 min) → comparison.

## 1. Setup

We use the **Video Games** subset of Amazon Reviews 2023:

- 2,000 products
- 39,796 continuous 1–5 ratings
- product metadata: price, store, categories, title, description, features
- 300 product images

We use PyTorch for the network and **skorch** for `fit` / `predict`, exactly like the NCF notebook.

In [1]:
!pip install skorch

In [2]:
import os
import warnings

warnings.filterwarnings("ignore", message="The given NumPy array is not writable")

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from skorch import NeuralNetRegressor
from skorch.dataset import ValidSplit
from torch import nn

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
torch.manual_seed(0)
np.random.seed(0)

BASE = "https://raw.githubusercontent.com/statmlben/CUHK-STAT3009/main/dataset/amazon-video-games"
LOCAL = next(
    (p for p in ["../dataset/amazon-video-games", "dataset/amazon-video-games"]
     if os.path.exists(f"{p}/ratings.csv")),
    None,
)

def read_csv(name, **kwargs):
    try:
        return pd.read_csv(f"{BASE}/{name}", **kwargs)
    except Exception:
        return pd.read_csv(f"{LOCAL}/{name}", **kwargs)

ratings = read_csv("ratings.csv")
items = read_csv("items.csv")
df = ratings.merge(items, on="item_id", how="left")

print("ratings:", len(df), "users:", df.user_id.nunique(), "items:", df.item_id.nunique())
print("rating mean:", f"{df.rating.mean():.4f}", "std:", f"{df.rating.std():.4f}")
display(df[["user_id", "item_id", "title", "price", "store", "rating"]].head())

ratings: 39796 users: 29209 items: 2000
rating mean: 4.2602 std: 1.1811


,user_id,item_id,title,price,store,rating
0,AGHDL7D2SU4TCK3YAA5T3ZMR7NVQ,B00C1TTF86,Battlefield 4 - Playstation 3,15.2200,Electronic Arts,5.0000
1,AHNZ26CG5WKOLGJR34G75I44SLQQ,B0080CAO9C,Pro Evolution Soccer 2013 - Xbox 360,20.2500,Konami,5.0000
2,AHNZBEHS3GE6KU4PXCL72WWEVUKA,B014E8HHOA,Wii U Fit Meter Green & White (Bulk Packaging),6.4900,Nintendo,5.0000
3,AGA3ZY7OQB64RI72KGMANS3JHHLQ,B012F20ZY6,New Super Mario Bros. U + New Super Luigi U - ...,60.9400,Nintendo,5.0000
4,AGHZRDHTFDE7JDEQV5U3PA5SZFEA,B01GYPB1HU,"(Pack of 2) Screen Protector for PS Vita 1000,...",8.4900,AKWOX,5.0000


## 2. Prepare the data

This section does all the data work once:

- randomly split the rating rows 80% / 20% into train and test;
- encode user and item IDs with `LabelEncoder`;
- keep only two continuous features: `price` and `user_count`;
- encode `store`, `main_category`, and `categories` as integer indices.

Each kind of input is kept as its **own array**, and a model receives them as named arguments of `forward`:

| key | shape | content |
|---|---|---|
| `ids` | `(N, 2)` | user index, item index |
| `cont` | `(N, 2)` | standardized `price`, `user_count` |
| `cat` | `(N, 3)` | `store`, `main_category`, `categories` indices |
| `text` | `(N, 384)` | text embedding (added in Section 6) |
| `image` | `(N, 512)` | image embedding (added in Section 7) |

skorch accepts a **dict** `X`; each key is passed to the argument of `forward` with the same name. A model is fitted on a dict that contains exactly the keys its `forward` takes.

In [3]:
# Random 80/20 split of rating rows.
train, test = train_test_split(df, test_size=0.2, random_state=3009)
train, test = train.copy(), test.copy()

# User activity is computed from training interactions only.
user_count = train.groupby("user_id").size().rename("user_count")
train = train.merge(user_count, on="user_id", how="left")
test = test.merge(user_count, on="user_id", how="left")
test["user_count"] = test["user_count"].fillna(0)

# Encode IDs and categories as integer indices (no ratings are used).
fields = ["user_id", "item_id", "store", "main_category", "categories"]
for col in fields:
    enc = LabelEncoder().fit(df[col].fillna("unknown").astype(str))
    train[col + "_idx"] = enc.transform(train[col].fillna("unknown").astype(str))
    test[col + "_idx"] = enc.transform(test[col].fillna("unknown").astype(str))

n_users, n_items, n_store, n_main, n_cat = (
    df[col].fillna("unknown").nunique() for col in fields
)

# Continuous features: log price and training-only user activity.
price_fill = train.price.median()
cont_cols = lambda d: np.column_stack([
    np.log1p(d.price.fillna(price_fill)), d.user_count])
scaler = StandardScaler().fit(cont_cols(train))
cont_train = scaler.transform(cont_cols(train))
cont_test = scaler.transform(cont_cols(test))

def make_X(d, cont):
    return {
        "ids": d[["user_id_idx", "item_id_idx"]].to_numpy().astype(np.float32),
        "cont": cont.astype(np.float32),
        "cat": d[["store_idx", "main_category_idx", "categories_idx"]].to_numpy().astype(np.float32),
    }

keys = lambda X, *names: {k: X[k] for k in names}   # pick the inputs a model needs

X_train = make_X(train, cont_train)
X_test = make_X(test, cont_test)

y_train = train.rating.to_numpy(dtype=np.float32).reshape(-1, 1)
y_test = test.rating.to_numpy(dtype=np.float32).reshape(-1, 1)
mu = float(y_train.mean())

print("train rows:", len(train), "test rows:", len(test))
print("n_users:", n_users, "n_items:", n_items)
print({k: v.shape for k, v in X_train.items()})
print("global mean:", f"{mu:.4f}")

train rows: 31836 test rows: 7960
n_users: 29209 n_items: 2000
{'ids': (31836, 2), 'cont': (31836, 2), 'cat': (31836, 3)}
global mean: 4.2602


## 3. MFNet baseline

This is the NCF `MFNet` as the base model: user and item embeddings, biases, and a dot product.

`skorch.NeuralNetRegressor` handles the training loop, validation split, and prediction.

In [4]:
class MFNet(nn.Module):
    def __init__(self, n_users, n_items, K=16, mu=0.0):
        super().__init__()
        torch.manual_seed(0)
        self.user_emb = nn.Embedding(n_users, K, max_norm=0.3)
        self.item_emb = nn.Embedding(n_items, K, max_norm=0.3)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)
        self.mu = nn.Parameter(torch.tensor([mu], dtype=torch.float32))

    def forward(self, ids):
        u = ids[:, 0].long()
        i = ids[:, 1].long()
        dot = (self.user_emb(u) * self.item_emb(i)).sum(dim=1, keepdim=True)
        return self.mu + self.user_bias(u) + self.item_bias(i) + dot


mf = NeuralNetRegressor(
    module=MFNet,
    module__n_users=n_users,
    module__n_items=n_items,
    module__mu=mu,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=0.05,
    max_epochs=50,
    batch_size=256,
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)
mf.fit(keys(X_train, "ids"), y_train)
mf_pred = mf.predict(keys(X_test, "ids"))

print("MFNet validation RMSE:", f"{np.sqrt(mf.history[-1, 'valid_loss']):.4f}")
print("MFNet test RMSE:", f"{root_mean_squared_error(y_test, mf_pred):.4f}")

MFNet validation RMSE: 1.1643
MFNet test RMSE: 1.1586


## 4. Add continuous features

Now add a small neural block for `price` and `user_count`:

$$
\widehat r_{ui}=\text{MFNet}(u,i)+g_{\text{cont}}(\text{price},\text{user\_count}).
$$

`MFNetCont` inherits `MFNet` and adds exactly one module. The base MF path is unchanged.

In [5]:
class MFNetCont(MFNet):
    def __init__(self, n_users, n_items, K=16, mu=0.0, d_cont=2):
        super().__init__(n_users, n_items, K, mu)
        self.cont = nn.Sequential(
            nn.Linear(d_cont, 16), nn.ReLU(), nn.Linear(16, 1)
        )

    def forward(self, ids, cont):
        return super().forward(ids) + self.cont(cont)


cont = NeuralNetRegressor(
    module=MFNetCont,
    module__n_users=n_users,
    module__n_items=n_items,
    module__mu=mu,
    module__d_cont=2,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=0.05,
    max_epochs=50,
    batch_size=256,
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)
cont.fit(keys(X_train, "ids", "cont"), y_train)
cont_pred = cont.predict(keys(X_test, "ids", "cont"))

print("MFNet + continuous validation RMSE:",
      f"{np.sqrt(cont.history[-1, 'valid_loss']):.4f}")
print("MFNet + continuous test RMSE:",
      f"{root_mean_squared_error(y_test, cont_pred):.4f}")

MFNet + continuous validation RMSE: 1.1685
MFNet + continuous test RMSE: 1.1617


### Other ways to fuse the continuous features

`MFNetCont` adds the continuous block **outside** the dot product:

$$
\widehat r_{ui}=\mu+a_u+b_i+p_u^\top q_i+g_{\text{cont}}(c_{ui}).
$$

The features can also be fused **inside** the interaction. Encode them as a vector $g$, concatenate it with the two embeddings, and let an MLP learn the interaction (the NCF style):

$$
\widehat r_{ui}=\mu+a_u+b_i+h_\theta\bigl([\,p_u,\;q_i,\;g(c_{ui})\,]\bigr).
$$

| fusion | where the features enter | interaction |
|---|---|---|
| additive (`MFNetCont`) | after the dot product, as a separate term | fixed dot product |
| concatenate (`MFNetContConcat`) | inside the MLP, next to $p_u$ and $q_i$ | learned MLP |

The concatenated version lets the features **change how $p_u$ and $q_i$ interact** (for example, price may matter more for some users), at the cost of dropping the explicit dot product.

In [6]:
class MFNetContConcat(nn.Module):
    def __init__(self, n_users, n_items, K=16, mu=0.0, d_cont=2, hidden=32):
        super().__init__()
        torch.manual_seed(0)
        self.user_emb = nn.Embedding(n_users, K, max_norm=0.3)
        self.item_emb = nn.Embedding(n_items, K, max_norm=0.3)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)
        self.mu = nn.Parameter(torch.tensor([mu], dtype=torch.float32))
        self.cont_enc = nn.Linear(d_cont, 8)                      # g(c): features -> 8-dim vector
        self.mlp = nn.Sequential(
            nn.Linear(2 * K + 8, hidden), nn.ReLU(), nn.Linear(hidden, 1)
        )

    def forward(self, ids, cont):
        u = ids[:, 0].long()
        i = ids[:, 1].long()
        z = torch.cat([self.user_emb(u), self.item_emb(i), self.cont_enc(cont)], dim=1)   # (B, 2K+8)
        return self.mu + self.user_bias(u) + self.item_bias(i) + self.mlp(z)


cont_concat = NeuralNetRegressor(
    module=MFNetContConcat,
    module__n_users=n_users,
    module__n_items=n_items,
    module__mu=mu,
    module__d_cont=2,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=0.05,
    max_epochs=50,
    batch_size=256,
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)
cont_concat.fit(keys(X_train, "ids", "cont"), y_train)
cont_concat_pred = cont_concat.predict(keys(X_test, "ids", "cont"))

print("concatenate fusion validation RMSE:",
      f"{np.sqrt(cont_concat.history[-1, 'valid_loss']):.4f}")
print("concatenate fusion test RMSE:",
      f"{root_mean_squared_error(y_test, cont_concat_pred):.4f}")
print("additive fusion test RMSE:   ",
      f"{root_mean_squared_error(y_test, cont_pred):.4f}")

concatenate fusion validation RMSE: 1.1880
concatenate fusion test RMSE: 1.1860
additive fusion test RMSE:    1.1617


## 5. Add categorical features

Next add one more block: `store`, `main_category`, and `categories`.

Each field gets its own `nn.Embedding` table. The three vectors are concatenated and passed through a small neural block:

$$
\widehat r_{ui}=\text{MFNet}(u,i)+g_{\text{cont}}(\cdot)+g_{\text{cat}}(\cdot).
$$

`MFNetCat` inherits `MFNetCont`; it only adds the categorical modules.

In [7]:
class MFNetCat(MFNetCont):
    def __init__(self, n_users, n_items, n_store, n_main, n_cat,
                 K=16, mu=0.0, d_cont=2, Kc=16):
        super().__init__(n_users, n_items, K, mu, d_cont)
        self.store_emb = nn.Embedding(n_store, Kc)
        self.main_emb = nn.Embedding(n_main, Kc)
        self.cat_emb = nn.Embedding(n_cat, Kc)
        self.cat_block = nn.Sequential(
            nn.Linear(3 * Kc, 16), nn.ReLU(), nn.Linear(16, 1)
        )

    def forward(self, ids, cont, cat):
        z = torch.cat([
            self.store_emb(cat[:, 0].long()),
            self.main_emb(cat[:, 1].long()),
            self.cat_emb(cat[:, 2].long()),
        ], dim=1)
        return super().forward(ids, cont) + self.cat_block(z)


cat = NeuralNetRegressor(
    module=MFNetCat,
    module__n_users=n_users,
    module__n_items=n_items,
    module__n_store=n_store,
    module__n_main=n_main,
    module__n_cat=n_cat,
    module__mu=mu,
    module__d_cont=2,
    module__Kc=16,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=0.05,
    max_epochs=50,
    batch_size=256,
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)
cat.fit(keys(X_train, "ids", "cont", "cat"), y_train)
cat_pred = cat.predict(keys(X_test, "ids", "cont", "cat"))

print("MFNet + continuous + categorical validation RMSE:",
      f"{np.sqrt(cat.history[-1, 'valid_loss']):.4f}")
print("MFNet + continuous + categorical test RMSE:",
      f"{root_mean_squared_error(y_test, cat_pred):.4f}")

MFNet + continuous + categorical validation RMSE: 1.1577
MFNet + continuous + categorical test RMSE: 1.1538


### In-class practice: fuse the categorical features differently

`MFNetCat` adds the categorical block as a separate **additive** term. As in Section 4, the other option is to fuse inside the interaction: concatenate the user embedding, the item embedding, and the three categorical embeddings, then let an MLP score the vector:

$$
\widehat r_{ui}=\mu+a_u+b_i+h_\theta\bigl([\,p_u,\;q_i,\;s_{\text{store}},\;s_{\text{main}},\;s_{\text{cat}}\,]\bigr).
$$

Complete `forward` in `MFNetCatConcat`, fit it, and compare it with `MFNetCat`.

1. Concatenate the five embeddings with `torch.cat(..., dim=1)`. What is the width of `z`?
2. Pass `z` through `self.mlp` and add the bias terms.
3. Fit it on `keys(X_train, "ids", "cat")` (this model has no `cont` input) and report validation and test RMSE.
4. Which fusion is better on validation? Why might the extra flexibility not help here?

In [8]:
class MFNetCatConcat(nn.Module):
    def __init__(self, n_users, n_items, n_store, n_main, n_cat,
                 K=16, mu=0.0, Kc=16, hidden=32):
        super().__init__()
        torch.manual_seed(0)
        self.user_emb = nn.Embedding(n_users, K, max_norm=0.3)
        self.item_emb = nn.Embedding(n_items, K, max_norm=0.3)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)
        self.mu = nn.Parameter(torch.tensor([mu], dtype=torch.float32))
        self.store_emb = nn.Embedding(n_store, Kc)
        self.main_emb = nn.Embedding(n_main, Kc)
        self.cat_emb = nn.Embedding(n_cat, Kc)
        self.mlp = nn.Sequential(
            nn.Linear(2 * K + 3 * Kc, hidden), nn.ReLU(), nn.Linear(hidden, 1)
        )

    def forward(self, ids, cat):
        u = ids[:, 0].long()
        i = ids[:, 1].long()
        # z = torch.cat([...], dim=1)
        # return ...
        ...

### Solution

In [9]:
class MFNetCatConcat(nn.Module):
    def __init__(self, n_users, n_items, n_store, n_main, n_cat,
                 K=16, mu=0.0, Kc=16, hidden=32):
        super().__init__()
        torch.manual_seed(0)
        self.user_emb = nn.Embedding(n_users, K, max_norm=0.3)
        self.item_emb = nn.Embedding(n_items, K, max_norm=0.3)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)
        self.mu = nn.Parameter(torch.tensor([mu], dtype=torch.float32))
        self.store_emb = nn.Embedding(n_store, Kc)
        self.main_emb = nn.Embedding(n_main, Kc)
        self.cat_emb = nn.Embedding(n_cat, Kc)
        self.mlp = nn.Sequential(
            nn.Linear(2 * K + 3 * Kc, hidden), nn.ReLU(), nn.Linear(hidden, 1)
        )

    def forward(self, ids, cat):
        u = ids[:, 0].long()
        i = ids[:, 1].long()
        z = torch.cat([
            self.user_emb(u), self.item_emb(i),
            self.store_emb(cat[:, 0].long()),
            self.main_emb(cat[:, 1].long()),
            self.cat_emb(cat[:, 2].long()),
        ], dim=1)                                    # (B, 2K + 3Kc)
        return self.mu + self.user_bias(u) + self.item_bias(i) + self.mlp(z)


cat_concat = NeuralNetRegressor(
    module=MFNetCatConcat,
    module__n_users=n_users,
    module__n_items=n_items,
    module__n_store=n_store,
    module__n_main=n_main,
    module__n_cat=n_cat,
    module__mu=mu,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=0.05,
    max_epochs=50,
    batch_size=256,
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)
cat_concat.fit(keys(X_train, "ids", "cat"), y_train)
cat_concat_pred = cat_concat.predict(keys(X_test, "ids", "cat"))

print("concatenate fusion validation RMSE:",
      f"{np.sqrt(cat_concat.history[-1, 'valid_loss']):.4f}")
print("concatenate fusion test RMSE:",
      f"{root_mean_squared_error(y_test, cat_concat_pred):.4f}")
print("additive fusion (MFNetCat) test RMSE:",
      f"{root_mean_squared_error(y_test, cat_pred):.4f}")

concatenate fusion validation RMSE: 1.1598
concatenate fusion test RMSE: 1.1571
additive fusion (MFNetCat) test RMSE: 1.1538


## 6. Add text

Text is computed here, not loaded from a precomputed file.

In Colab, install HuggingFace `transformers` once:

```python
!pip -q install transformers
```

We use the `feature-extraction` pipeline with `all-MiniLM-L6-v2`: text goes in, one 384-dim vector per token comes out. Averaging over the tokens gives one vector per product, built from the title, description, and features. The vectors are stored as a new input `X_train["text"]`.

The new network block is `MFNetText`, which inherits `MFNetCat` and adds one text projection + one neural block.

In [10]:
# Colab: uncomment the next line once if transformers is missing.
# !pip -q install transformers

from transformers import pipeline

embed = pipeline("feature-extraction", model="sentence-transformers/all-MiniLM-L6-v2")

texts = (items.title.fillna("") + " " + items.description.fillna("")
         + " " + items.features.fillna("")).tolist()

# each output is (1, n_tokens, 384); average over tokens -> (384,)
outputs = embed(texts, batch_size=64, truncation=True, max_length=256)
text_vectors = np.array([np.mean(o[0], axis=0) for o in outputs])
d_text = text_vectors.shape[1]

# one row of text_vectors per item; look up each rating row by its item_id
row_of = pd.Series(range(len(items)), index=items.item_id)
X_train["text"] = text_vectors[row_of[train.item_id].to_numpy()].astype(np.float32)
X_test["text"] = text_vectors[row_of[test.item_id].to_numpy()].astype(np.float32)
print("text embedding:", X_train["text"].shape)

text embedding: (31836, 384)


In [11]:
class MFNetText(MFNetCat):
    def __init__(self, n_users, n_items, n_store, n_main, n_cat,
                 K=16, mu=0.0, d_cont=2, Kc=16, d_text=384, Kt=32):
        super().__init__(n_users, n_items, n_store, n_main, n_cat,
                         K, mu, d_cont, Kc)
        self.text_proj = nn.Linear(d_text, Kt)
        self.text_block = nn.Sequential(
            nn.Linear(Kt, 16), nn.ReLU(), nn.Linear(16, 1)
        )

    def forward(self, ids, cont, cat, text):
        return super().forward(ids, cont, cat) + self.text_block(self.text_proj(text))


text = NeuralNetRegressor(
    module=MFNetText,
    module__n_users=n_users,
    module__n_items=n_items,
    module__n_store=n_store,
    module__n_main=n_main,
    module__n_cat=n_cat,
    module__mu=mu,
    module__d_cont=2,
    module__Kc=16,
    module__d_text=d_text,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=0.05,
    max_epochs=50,
    batch_size=256,
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)
text.fit(keys(X_train, "ids", "cont", "cat", "text"), y_train)
text_pred = text.predict(keys(X_test, "ids", "cont", "cat", "text"))

print("MFNet + continuous + categorical + text validation RMSE:",
      f"{np.sqrt(text.history[-1, 'valid_loss']):.4f}")
print("MFNet + continuous + categorical + text test RMSE:",
      f"{root_mean_squared_error(y_test, text_pred):.4f}")

MFNet + continuous + categorical + text validation RMSE: 1.1710
MFNet + continuous + categorical + text test RMSE: 1.1668


## 7. Add image

Only 300 products have a local image. We encode them here with a pretrained ResNet18 through the HuggingFace `image-feature-extraction` pipeline, not from a CSV.

The image block is `MFNetImage`, which inherits `MFNetText`. We evaluate on the subset of products that have an image, comparing the text model with and without the image block.

In [12]:
from transformers import pipeline

embed_image = pipeline("image-feature-extraction", model="microsoft/resnet-18")

# Only 300 products have an image; image_path is relative to the dataset folder.
image_items = items[items.image_path.notna()]
paths = [f"{LOCAL or BASE}/{p}" for p in image_items.image_path]

# pool=True averages the 7x7 feature map: each output is (1, 512, 1, 1) -> flatten to 512
image_vectors = np.array(embed_image(paths, batch_size=32, pool=True)).reshape(len(paths), -1)
image_vectors /= np.linalg.norm(image_vectors, axis=1, keepdims=True)   # unit length, like the text vectors
image_emb = pd.DataFrame(image_vectors, index=image_items.item_id)
d_image = image_emb.shape[1]
print("image embedding:", image_emb.shape)

image embedding: (300, 512)


In [13]:
# keep only the rating rows whose product has an image
train_img = train.item_id.isin(image_emb.index).to_numpy()
test_img = test.item_id.isin(image_emb.index).to_numpy()

X_train_img = {k: v[train_img] for k, v in X_train.items()}
X_test_img = {k: v[test_img] for k, v in X_test.items()}
X_train_img["image"] = image_emb.loc[train.item_id[train_img]].to_numpy().astype(np.float32)
X_test_img["image"] = image_emb.loc[test.item_id[test_img]].to_numpy().astype(np.float32)
y_train_img, y_test_img = y_train[train_img], y_test[test_img]
print("image subset:", {k: v.shape for k, v in X_train_img.items()})

image subset: {'ids': (4711, 2), 'cont': (4711, 2), 'cat': (4711, 3), 'text': (4711, 384), 'image': (4711, 512)}


In [14]:
class MFNetImage(MFNetText):
    def __init__(self, n_users, n_items, n_store, n_main, n_cat,
                 K=16, mu=0.0, d_cont=2, Kc=16, d_text=384, Kt=32,
                 d_image=512, Ki=32):
        super().__init__(n_users, n_items, n_store, n_main, n_cat,
                         K, mu, d_cont, Kc, d_text, Kt)
        self.image_proj = nn.Linear(d_image, Ki)
        self.image_block = nn.Sequential(
            nn.Linear(Ki, 16), nn.ReLU(), nn.Linear(16, 1)
        )

    def forward(self, ids, cont, cat, text, image):
        base = super().forward(ids, cont, cat, text)
        return base + self.image_block(self.image_proj(image))


text_img = NeuralNetRegressor(
    module=MFNetText,
    module__n_users=n_users,
    module__n_items=n_items,
    module__n_store=n_store,
    module__n_main=n_main,
    module__n_cat=n_cat,
    module__mu=mu,
    module__d_cont=2,
    module__Kc=16,
    module__d_text=d_text,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=0.05,
    max_epochs=50,
    batch_size=256,
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)
text_keys = ("ids", "cont", "cat", "text")
text_img.fit(keys(X_train_img, *text_keys), y_train_img)

image_model = NeuralNetRegressor(
    module=MFNetImage,
    module__n_users=n_users,
    module__n_items=n_items,
    module__n_store=n_store,
    module__n_main=n_main,
    module__n_cat=n_cat,
    module__mu=mu,
    module__d_cont=2,
    module__Kc=16,
    module__d_text=d_text,
    module__d_image=d_image,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=0.05,
    max_epochs=50,
    batch_size=256,
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)
image_model.fit(X_train_img, y_train_img)

print("image subset, text only test RMSE:",
      f"{root_mean_squared_error(y_test_img, text_img.predict(keys(X_test_img, *text_keys))):.4f}")
print("image subset, + image test RMSE:",
      f"{root_mean_squared_error(y_test_img, image_model.predict(X_test_img)):.4f}")

image subset, text only test RMSE: 1.1379
image subset, + image test RMSE: 1.1346


## 8. Compare the models

The comparison table shows the contribution of each block. All models use the same split, the same optimizer, and the same epoch budget.

In [15]:
def test_rmse(model, X, y):
    return root_mean_squared_error(y, model.predict(X))

results = pd.DataFrame([
    {"model": "MFNet", "side info": "IDs only",
     "test RMSE": test_rmse(mf, keys(X_test, "ids"), y_test)},
    {"model": "+ continuous", "side info": "price + user_count",
     "test RMSE": test_rmse(cont, keys(X_test, "ids", "cont"), y_test)},
    {"model": "+ categorical", "side info": "store + category",
     "test RMSE": test_rmse(cat, keys(X_test, "ids", "cont", "cat"), y_test)},
    {"model": "+ text", "side info": "MiniLM description",
     "test RMSE": test_rmse(text, keys(X_test, "ids", "cont", "cat", "text"), y_test)},
]).set_index("model")

display(results.round(4))

print("image subset, text only:",
      f"{test_rmse(text_img, keys(X_test_img, *text_keys), y_test_img):.4f}")
print("image subset, + image:  ",
      f"{test_rmse(image_model, X_test_img, y_test_img):.4f}")

,side info,test RMSE
model,,
MFNet,IDs only,1.1586
+ continuous,price + user_count,1.1617
+ categorical,store + category,1.1538
+ text,MiniLM description,1.1668


image subset, text only: 1.1379
image subset, + image:   1.1346


The table is the teaching point: each section adds one neural block, and the test RMSE shows what that block contributes. Image is evaluated on a smaller subset, so its numbers are not directly comparable with the main table.

## 9. Exercises

1. **New items.** Hold out 20% of products instead of random rows. Which blocks still help?
2. **Continuous block.** Replace the 16-unit hidden layer with a direct `nn.Linear(2, 1)`. Does the extra depth matter?
3. **Categorical block.** Add one more categorical field, or increase `Kc`. Does validation improve?
4. **Text encoder.** Replace MiniLM with a larger HuggingFace embedding model. Does the text block become more useful?
5. **Image encoder.** Replace ResNet18 with a ViT (e.g. `google/vit-base-patch16-224`). Does the image block improve the image subset?
6. **Leakage audit.** Add `average_rating` as a feature. Why is it invalid when computed on all ratings?

For each exercise, keep the split fixed and change one thing at a time.

## Summary

```text
Amazon Video Games subset
  → prepare data + random split
  → MFNet (NCF base) with skorch
  → + continuous block
  → + categorical block
  → + text block
  → + image block
  → RMSE comparison
```

The base MF path never changes. Each section adds one small neural module and one named input. That makes it clear what each side-information type contributes.